# Oorspronkelijke hoogte stinswier de Worf in Lies op Terschelling. 

Collega, Jan van Doesburg, wil graag weten hoe hoe hoog stinswier de Worf in Lies op Terschelling mogelijk is geweest bij de aanleg. 

“Het onderzoek heeft uitgewezen dat de stinswier de Worf in de dertiende eeuw is gesticht. Dit kan worden afgeleid uit het vondstmateriaal, maar vooral uit de ¹⁴C-datering van een monster uit de onderste grachtvulling. De heuvel is hoogstwaarschijnlijk in één keer is opgeworpen en bestaat uit grond die bij het uitgraven van de 12 tot 14 m brede en 1.60-1.80 m diepe gracht vrijkwam. De huidige heuvel heeft een hoogte van 3.1 m tov NAP, maar was oorspronkelijk waarschijnlijk hoger. Op basis van een berekening van de grond die bij het uitgraven van de gracht is vrijgekomen, zou de heuvel oorspronkelijk ?? hoog kunnen zijn geweest. “


## 1.	Huidige situatie
### 1.1 Bepalen huidige volume van de heuvel
Als eerste stap is gezamenlijk een contour ingetekend van het verhoogde deel “de berg” en een contour van de omliggende greppel (figuur 1). Het is duidelijk dat de greppel niet meer intact is en nu de contour van het huidige kavel heeft. Dit is ingetekend op AHN5 en er is gebruik gemaakt van het Digital Terrain Model (DTM), dat wil zeggen het gereconstrueerde maaiveld. 
 
<img src=".//img//ahn_5_contour.png" alt="" style="display: inline-block; height: 1000; width: 1250; vertical-align: text-bottom; margin: 0 0.25rem;" />

Figuur 1 - huidige uitsnede AHN 5

Op basis van de huidige situatie is het volume van de huidige heuvel bepaald op 939.02 m³. 
De ingetekende contour is gebruikt als input voor de berekening. Als eerste stap is er een buffer van 1 meter om de ingetekende contour gelegd. Vervolgens is binnen deze buffer de gemiddelde waarde berekend van het AHN en zijn voor de zekerheid de minimale 2% en maximale 2% niet meegenomen. Op basis van deze berekening is de NAP hoogte bepaald van de voet van de heuvel. Vervolgens zijn hier de waardes van de heuvel afgetrokken waardoor per cel bekend is hoe hoog deze is tov de voet van de heuvel. 
Om op basis daarvan het volume te bepalen zijn alle waardes, na aftrek van de gemiddelde NAP hoogte van de voet binnen de contour, bij elkaar opgeteld en gedeeld door 4 (aangezien 1 cell 0.5 x 0.5 meter is). 


#### Libraries

In [1]:
import rasterio
import numpy as np
from pathlib import Path
import geopandas as gpd
import pandas as pd
from rasterio.features import geometry_mask
from rasterio.mask import mask
from rasterio.transform import xy
from shapely.geometry import mapping
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


#### Downloaden AHN TIF

In [2]:


url_dem = "https://fsn1.your-objectstorage.com/hwh-ahn/AHN5_KM/02a_DTM_50cm/AHN5_M_151000_600000.TIF"
output_file = "./data/AHN5_M_151000_600000.TIF"

# Retry failed connections
session = requests.Session()

retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=2
)

session.mount("https://", HTTPAdapter(max_retries=retry))

# Stream the file instead of loading it completely into memory
with session.get(url_dem, stream=True, timeout=(10, 120)) as r:
    r.raise_for_status()

    with open(output_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Download complete")

Download complete


#### Analyse 1 bepalen van volume op basis van contour

In [3]:
# Variabelen instellen
ahn_file = ".//data//AHN5_M_151000_600000.TIF"
gpkg_contour = ".//data//de_worf.gpkg"
layer_contour =  "contour_mound" 


In [4]:
# Calculate the volume based on contour

# ============================================================
# SETTINGS
# ============================================================

dem_path = Path(ahn_file)
gpkg_path = Path(gpkg_contour)

# Set to None to read the first layer in the GeoPackage
polygon_layer = "contour_mound"

output_folder = Path("mound_volume_results")

# Width of the terrain sampling ring outside the mound, in metres
ring_inner_buffer = 1.0
ring_outer_buffer = 5.0

# Remove extreme elevations in the reference ring before fitting
lower_percentile = 2
upper_percentile = 98

# Set negative mound heights to zero
use_positive_heights_only = True


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def fit_reference_plane(x, y, z):
    """
    Fit the plane:

        z = a*x + b*y + c

    using least squares.
    """
    design_matrix = np.column_stack(
        [
            x,
            y,
            np.ones_like(x),
        ]
    )

    coefficients, _, _, _ = np.linalg.lstsq(
        design_matrix,
        z,
        rcond=None,
    )

    return coefficients


def evaluate_plane(x, y, coefficients):
    """
    Calculate plane elevations for x and y coordinates.
    """
    a, b, c = coefficients
    return a * x + b * y + c


# ============================================================
# PREPARE OUTPUT FOLDER
# ============================================================

output_folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# READ THE MOUND POLYGON
# ============================================================

if polygon_layer is None:
    polygon_layers = gpd.list_layers(gpkg_path)

    if polygon_layers.empty:
        raise ValueError("No layers were found in the GeoPackage.")

    polygon_layer = polygon_layers.iloc[0]["name"]

print(f"Reading polygon layer: {polygon_layer}")

mounds = gpd.read_file(
    gpkg_path,
    layer=polygon_layer,
)

if mounds.empty:
    raise ValueError("The polygon layer contains no features.")

if mounds.crs is None:
    raise ValueError("The mound polygon has no CRS.")

# Repair invalid geometries where possible
mounds["geometry"] = mounds.geometry.make_valid()

# Remove empty geometries
mounds = mounds[
    mounds.geometry.notna()
    & ~mounds.geometry.is_empty
].copy()

if mounds.empty:
    raise ValueError("No valid mound geometries remain.")

# Combine all polygons into one mound geometry
mound_geometry = mounds.geometry.union_all()


# ============================================================
# OPEN THE DEM
# ============================================================

with rasterio.open(dem_path) as src:

    if src.crs is None:
        raise ValueError("The DEM has no CRS.")

    print(f"DEM CRS: {src.crs}")
    print(f"Polygon CRS: {mounds.crs}")

    # --------------------------------------------------------
    # CRS handling
    # --------------------------------------------------------

    # AHN uses Amersfoort / RD New (EPSG:28992) horizontally.
    # Elevations are relative to NAP.
    #
    # Some AHN GeoTIFFs encode this as a LOCAL_CS:
    # "Amersfoort / RD New + NAP height".
    # We therefore don't try to reproject to src.crs.

    if mounds.crs.to_epsg() != 28992:
        print("Reprojecting mound polygon to EPSG:28992")
        mounds = mounds.to_crs(epsg=28992)

    mound_geometry = mounds.geometry.union_all()

    # --------------------------------------------------------
    # Create the reference ring
    # --------------------------------------------------------

    inner_boundary = mound_geometry.buffer(ring_inner_buffer)
    outer_boundary = mound_geometry.buffer(ring_outer_buffer)

    reference_ring = outer_boundary.difference(inner_boundary)
    
    if reference_ring.is_empty:
        raise ValueError("The reference ring could not be created.")

    # --------------------------------------------------------
    # Crop the DEM to the outer ring
    # --------------------------------------------------------

    cropped_dem, cropped_transform = mask(
        src,
        [mapping(outer_boundary)],
        crop=True,
        filled=False,
    )

    cropped_dem = cropped_dem[0]

    # Convert masked raster to float with NaN
    dem_array = cropped_dem.filled(np.nan).astype(np.float64)

    rows, cols = dem_array.shape

    # --------------------------------------------------------
    # Calculate the centre coordinate of each raster cell
    # --------------------------------------------------------

    row_indices, col_indices = np.indices((rows, cols))

    x_coordinates, y_coordinates = xy(
        cropped_transform,
        row_indices.ravel(),
        col_indices.ravel(),
        offset="center",
    )

    x_coordinates = np.asarray(
        x_coordinates,
        dtype=np.float64
    ).reshape(rows, cols)

    y_coordinates = np.asarray(
        y_coordinates,
        dtype=np.float64
    ).reshape(rows, cols)

    # --------------------------------------------------------
    # Create raster masks
    # --------------------------------------------------------

    mound_mask = geometry_mask(
        [mapping(mound_geometry)],
        out_shape=dem_array.shape,
        transform=cropped_transform,
        invert=True,
        all_touched=False,
    )

    ring_mask = geometry_mask(
        [mapping(reference_ring)],
        out_shape=dem_array.shape,
        transform=cropped_transform,
        invert=True,
        all_touched=False,
    )

    valid_dem = np.isfinite(dem_array)

    mound_valid = mound_mask & valid_dem
    ring_valid = ring_mask & valid_dem

    if mound_valid.sum() == 0:
        raise ValueError(
            "No valid DEM cells were found inside the mound polygon."
        )

    if ring_valid.sum() < 3:
        raise ValueError(
            "Too few valid DEM cells were found in the reference ring."
        )

    # --------------------------------------------------------
    # Extract reference-ring samples
    # --------------------------------------------------------

    x_ring = x_coordinates[ring_valid]
    y_ring = y_coordinates[ring_valid]
    z_ring = dem_array[ring_valid]

    # Remove extreme values from the reference sample
    lower_limit = np.nanpercentile(
        z_ring,
        lower_percentile,
    )

    upper_limit = np.nanpercentile(
        z_ring,
        upper_percentile,
    )

    keep = (
        (z_ring >= lower_limit)
        & (z_ring <= upper_limit)
    )

    x_ring = x_ring[keep]
    y_ring = y_ring[keep]
    z_ring = z_ring[keep]

    if len(z_ring) < 3:
        raise ValueError(
            "Too few reference cells remain after filtering."
        )

    # --------------------------------------------------------
    # Improve numerical stability by using local coordinates
    # --------------------------------------------------------

    x_origin = np.mean(x_ring)
    y_origin = np.mean(y_ring)

    x_ring_local = x_ring - x_origin
    y_ring_local = y_ring - y_origin

    plane_coefficients = fit_reference_plane(
        x_ring_local,
        y_ring_local,
        z_ring,
    )

    # Evaluate the reference plane across the cropped raster
    reference_surface = evaluate_plane(
        x_coordinates - x_origin,
        y_coordinates - y_origin,
        plane_coefficients,
    )

    # --------------------------------------------------------
    # Calculate mound height
    # --------------------------------------------------------

    height_above_reference = dem_array - reference_surface

    # Outside the mound, write NaN
    height_above_reference[~mound_valid] = np.nan

    if use_positive_heights_only:
        volume_heights = np.where(
            height_above_reference > 0,
            height_above_reference,
            0,
        )
    else:
        volume_heights = height_above_reference.copy()

    # --------------------------------------------------------
    # Calculate cell area and volume
    # --------------------------------------------------------

    cell_width = abs(cropped_transform.a)
    cell_height = abs(cropped_transform.e)
    cell_area = cell_width * cell_height

    volume_m3 = np.nansum(volume_heights) * cell_area

    mound_area_raster_m2 = mound_valid.sum() * cell_area
    mound_area_vector_m2 = mound_geometry.area

    positive_heights = height_above_reference[
        np.isfinite(height_above_reference)
        & (height_above_reference > 0)
    ]

    all_heights = height_above_reference[
        np.isfinite(height_above_reference)
    ]

    # --------------------------------------------------------
    # Calculate fit quality
    # --------------------------------------------------------

    fitted_ring_elevations = evaluate_plane(
        x_ring_local,
        y_ring_local,
        plane_coefficients,
    )

    ring_residuals = z_ring - fitted_ring_elevations

    ring_rmse = np.sqrt(
        np.mean(ring_residuals ** 2)
    )

    # --------------------------------------------------------
    # Export height-difference raster
    # --------------------------------------------------------

    output_raster = (
        output_folder / "mound_height_above_reference.tif"
    )

    output_profile = src.profile.copy()
    output_profile.update(
        driver="GTiff",
        height=height_above_reference.shape[0],
        width=height_above_reference.shape[1],
        transform=cropped_transform,
        count=1,
        dtype="float32",
        nodata=-9999.0,
        compress="deflate",
    )

    raster_output_array = np.where(
        np.isfinite(height_above_reference),
        height_above_reference,
        -9999.0,
    ).astype(np.float32)

    with rasterio.open(
        output_raster,
        "w",
        **output_profile,
    ) as dst:
        dst.write(raster_output_array, 1)


# ============================================================
# CREATE SUMMARY
# ============================================================

a, b, c = plane_coefficients

summary = {
    "dem": str(dem_path),
    "polygon_file": str(gpkg_path),
    "polygon_layer": polygon_layer,
    "cell_width_m": cell_width,
    "cell_height_m": cell_height,
    "cell_area_m2": cell_area,
    "mound_cells": int(mound_valid.sum()),
    "mound_area_raster_m2": mound_area_raster_m2,
    "mound_area_vector_m2": mound_area_vector_m2,
    "reference_ring_cells": int(len(z_ring)),
    "reference_ring_rmse_m": ring_rmse,
    "minimum_height_m": np.nanmin(all_heights),
    "maximum_height_m": np.nanmax(all_heights),
    "mean_height_m": np.nanmean(all_heights),
    "mean_positive_height_m": (
        np.nanmean(positive_heights)
        if len(positive_heights) > 0
        else 0
    ),
    "positive_volume_m3": volume_m3,
    "plane_x_coefficient": a,
    "plane_y_coefficient": b,
    "plane_intercept": c,
    "x_origin": x_origin,
    "y_origin": y_origin,
    "ring_inner_buffer_m": ring_inner_buffer,
    "ring_outer_buffer_m": ring_outer_buffer,
}

summary_df = pd.DataFrame([summary])

output_csv = output_folder / "mound_volume_summary.csv"
summary_df.to_csv(
    output_csv,
    index=False,
)

Reading polygon layer: contour_mound
DEM CRS: LOCAL_CS["Amersfoort / RD New + NAP height",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Polygon CRS: EPSG:28992


#### PRINT RESULTS - Analyse 1 bepalen van volume op basis van contour

In [5]:
print()
print("Calculation completed")
print("---------------------")
print(f"Vector mound area: {mound_area_vector_m2:,.2f} m²")
print(f"Raster mound area: {mound_area_raster_m2:,.2f} m²")
print(f"Maximum mound height: {np.nanmax(all_heights):,.2f} m")
#print(f"Mean mound height: {np.nanmean(all_heights):,.2f} m")
print(f"Estimated positive volume: {volume_m3:,.2f} m³")
#print(f"Reference-plane RMSE: {ring_rmse:,.3f} m")
print()
print(f"Summary written to: {output_csv}")
print(f"Height raster written to: {output_raster}")


Calculation completed
---------------------
Vector mound area: 803.70 m²
Raster mound area: 804.25 m²
Maximum mound height: 2.53 m
Estimated positive volume: 939.02 m³

Summary written to: mound_volume_results\mound_volume_summary.csv
Height raster written to: mound_volume_results\mound_height_above_reference.tif


## 1.2 Bepalen van plateau 
Uitgaande dat de top van de heuvel oorspronkelijk is afgevlakt is door in QGIS gebruik te maken van de profile tool (https://plugins.qgis.org/plugins/profiletool/ ) een profiel gemaakt van de heuvel (figuur 2).  

<img src=".//img//map.png" alt="" style="display: inline-block; height: 1000; width: 1250; vertical-align: text-bottom; margin: 0 0.25rem;" />

<img src=".//img//profiel.png" alt="" style="display: inline-block; height: 1000; width: 1250; vertical-align: text-bottom; margin: 0 0.25rem;" />

 
Figuur 2 – hoogte profiel van heuvel

Op basis van dit profiel is de aanname dat de hellingen van de heuvel tot ongeveer 8 meter vanuit de contour hebben gelopen. 


## 2.	Gereconstrueerde situatie

### 2.1 Volume van de gracht
Op basis van boringen rondom de heuvel is gereconstrueerd dat de gracht 12 tot 14 meter breed is geweest en een maximale diepte van 1.60 tot 1.80 meter heeft gehad. De dichtheid van de boringen is niet hoog genoeg om exact de helling van de gracht te bepalen. Voor de berekening is dan uitgegaan van 13 meter doorsnede en 1.70 meter diepte. Als aanname is gedaan dat de maximale diepte op een kwart vanuit de randen van de gracht werd bereikt en dat men in een evenredige hoek de gracht uitgroef. Dit zou neerkomen op een hoek van 14.6° wat een volume van 1592m3 geeft wanneer berekend vanuit de ingetekende contour. Echter, wanneer een steilere hoek zou zijn gehanteerd kan het al enorm verschillen. Een hoek van 25° geeft 2838 m3 en 30° geeft 3514 m3.


#### Analyse 2 Volume gracht - parameters 

In [7]:
# Total width of the ditch in metres
ditch_width = 13.0

# HARD maximum depth of the ditch in metres
ditch_depth = 1.7 

# Angle of the ditch sides measured from horizontal.
#
# Examples:
# 10 = gentle slope
# 20 = moderate slope
# 45 = steep slope
# 89 = almost vertical
#
# IMPORTANT:
# ditch_depth remains the maximum depth regardless of angle.
ditch_angle_degrees = 20

#### Analyse 2 Volume gracht - uitvoeren analyse

In [8]:
from pathlib import Path
import math

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.windows import from_bounds, transform as window_transform
from scipy.ndimage import distance_transform_edt


# ============================================================
# SETTINGS
# ============================================================

gpkg = gpkg_contour
layer = layer_contour

existing_dem_path = Path(ahn_file)




output_folder = Path("ditch_results")

relative_output_tif = (
    output_folder / "reconstructed_ditch_relative.tif"
)

absolute_output_tif = (
    output_folder / "reconstructed_ditch_absolute.tif"
)

combined_output_tif = (
    output_folder / "existing_dem_with_ditch.tif"
)

nodata_value = -9999.0


# ============================================================
# VALIDATE SETTINGS
# ============================================================

if ditch_width <= 0:
    raise ValueError(
        "ditch_width must be greater than zero."
    )

if ditch_depth <= 0:
    raise ValueError(
        "ditch_depth must be greater than zero."
    )

if not 0 < ditch_angle_degrees < 90:
    raise ValueError(
        "ditch_angle_degrees must be between 0 and 90 degrees."
    )


# ============================================================
# CALCULATE DITCH GEOMETRY
# ============================================================

half_width = ditch_width / 2.0

# Vertical change per horizontal metre
slope = math.tan(
    math.radians(ditch_angle_degrees)
)

# Horizontal distance required for one side slope
# to reach the maximum allowed depth.
distance_to_max_depth = (
    ditch_depth / slope
)

# ------------------------------------------------------------
# Determine actual ditch shape
# ------------------------------------------------------------

if distance_to_max_depth < half_width:

    # Maximum depth is reached before the middle of the ditch.
    # The remaining part therefore becomes a flat bottom.

    actual_max_depth = ditch_depth

    flat_bottom_half_width = (
        half_width - distance_to_max_depth
    )

    flat_bottom_width = (
        2.0 * flat_bottom_half_width
    )

    geometry_method = (
        "Side slopes capped at maximum depth; flat bottom"
    )

else:

    # The two side slopes meet before reaching the specified
    # maximum depth. The ditch is therefore V-shaped.

    actual_max_depth = (
        half_width * slope
    )

    flat_bottom_width = 0.0

    geometry_method = (
        "V-shaped ditch; maximum depth not reached"
    )


print("Ditch geometry")
print("--------------")
print(
    f"Width                    : "
    f"{ditch_width:.2f} m"
)
print(
    f"Side angle               : "
    f"{ditch_angle_degrees:.2f}°"
)
print(
    f"Maximum allowed depth    : "
    f"{ditch_depth:.2f} m"
)
print(
    f"Actual maximum depth     : "
    f"{actual_max_depth:.2f} m"
)
print(
    f"Distance to max depth    : "
    f"{distance_to_max_depth:.3f} m"
)
print(
    f"Flat bottom width        : "
    f"{flat_bottom_width:.3f} m"
)


# ============================================================
# READ CONTOUR
# ============================================================

if layer is None:

    layers = gpd.list_layers(gpkg)

    if layers.empty:
        raise ValueError(
            "No layers were found in the GeoPackage."
        )

    layer = layers.iloc[0]["name"]


gdf = gpd.read_file(
    gpkg,
    layer=layer,
)


if gdf.empty:
    raise ValueError(
        "The selected contour layer contains no features."
    )


if gdf.crs is None:
    raise ValueError(
        "The contour layer has no CRS."
    )


# Remove empty geometries
gdf = gdf[
    gdf.geometry.notna()
    & ~gdf.geometry.is_empty
].copy()


if gdf.empty:
    raise ValueError(
        "No valid geometries remain."
    )


# Repair invalid geometries if necessary
try:
    gdf["geometry"] = gdf.geometry.make_valid()
except AttributeError:
    gdf["geometry"] = gdf.geometry.buffer(0)


# ============================================================
# OPEN EXISTING DEM
# ============================================================

output_folder.mkdir(
    parents=True,
    exist_ok=True,
)


with rasterio.open(existing_dem_path) as src:

    if src.crs is None:
        raise ValueError(
            "The existing DEM has no CRS."
        )


    # --------------------------------------------------------
    # CRS HANDLING
    # --------------------------------------------------------
    #
    # AHN uses Amersfoort / RD New horizontally and NAP
    # vertically.
    #
    # Some AHN files report this as a LOCAL_CS instead of
    # EPSG:28992.
    #
    # We therefore regard the horizontal coordinates as
    # RD New.
    # --------------------------------------------------------

    horizontal_crs = "EPSG:28992"

    print()
    print(f"DEM CRS     : {src.crs}")
    print(f"Contour CRS : {gdf.crs}")


    # Make sure contour is RD New
    if gdf.crs.to_epsg() != 28992:

        print(
            "Reprojecting contour to EPSG:28992"
        )

        gdf = gdf.to_crs(
            horizontal_crs
        )


    # --------------------------------------------------------
    # COMBINE CONTOUR GEOMETRIES
    # --------------------------------------------------------

    try:
        polygon = gdf.geometry.union_all()

    except AttributeError:
        polygon = gdf.geometry.union_all()


    if polygon.is_empty:
        raise ValueError(
            "The combined contour geometry is empty."
        )


    # --------------------------------------------------------
    # CREATE DITCH POLYGON
    # --------------------------------------------------------
    #
    # Input contour = inner edge of ditch.
    #
    # Ditch extends OUTWARD by ditch_width.
    # --------------------------------------------------------

    outer_polygon = polygon.buffer(
        ditch_width
    )

    ditch_polygon = (
        outer_polygon.difference(polygon)
    )


    if ditch_polygon.is_empty:
        raise ValueError(
            "The ditch polygon could not be created."
        )


    # Approximate ditch centreline
    centreline = polygon.buffer(
        half_width
    ).boundary

    centreline_length = (
        centreline.length
    )


    # --------------------------------------------------------
    # CHECK OVERLAP WITH DEM
    # --------------------------------------------------------

    from shapely.geometry import box

    dem_bounds_polygon = box(
        src.bounds.left,
        src.bounds.bottom,
        src.bounds.right,
        src.bounds.top,
    )


    if not outer_polygon.intersects(
        dem_bounds_polygon
    ):
        raise ValueError(
            "The reconstructed ditch does not overlap the DEM."
        )


    # --------------------------------------------------------
    # CREATE WINDOW MATCHING DEM GRID
    # --------------------------------------------------------

    minx, miny, maxx, maxy = (
        outer_polygon.bounds
    )


    window = from_bounds(
        minx,
        miny,
        maxx,
        maxy,
        transform=src.transform,
    )


    # Expand to whole raster cells
    window = (
        window
        .round_offsets()
        .round_lengths()
    )


    # Full DEM window
    full_window = rasterio.windows.Window(
        col_off=0,
        row_off=0,
        width=src.width,
        height=src.height,
    )


    # Clip requested window to DEM
    window = window.intersection(
        full_window
    )


    local_transform = window_transform(
        window,
        src.transform,
    )


    # --------------------------------------------------------
    # READ EXISTING TERRAIN
    # --------------------------------------------------------

    existing_dem = src.read(
        1,
        window=window,
        masked=True,
    )


    existing_dem_array = (
        existing_dem
        .filled(np.nan)
        .astype(np.float32)
    )


    raster_height, raster_width = (
        existing_dem_array.shape
    )


    pixel_width = abs(
        local_transform.a
    )

    pixel_height = abs(
        local_transform.e
    )


    # --------------------------------------------------------
    # RASTERIZE DITCH
    # --------------------------------------------------------

    ditch_mask = rasterize(
        [(ditch_polygon, 1)],
        out_shape=(
            raster_height,
            raster_width,
        ),
        transform=local_transform,
        fill=0,
        all_touched=False,
        dtype=np.uint8,
    )


    # --------------------------------------------------------
    # DISTANCE TO NEAREST DITCH EDGE
    # --------------------------------------------------------
    #
    # The ditch polygon is a ring.
    #
    # Cells outside the ditch and inside the mound are 0.
    #
    # distance_transform_edt therefore calculates the
    # distance of every ditch cell to its nearest edge.
    # --------------------------------------------------------

    distance_to_edge = (
        distance_transform_edt(
            ditch_mask,
            sampling=(
                pixel_height,
                pixel_width,
            ),
        )
    )


    # ========================================================
    # CALCULATE RELATIVE DITCH DEPTH
    # ========================================================
    #
    # Start at 0 m at both edges.
    #
    # Move downward according to the specified side angle.
    #
    # IMPORTANT:
    #
    # Depth is capped at ditch_depth.
    #
    # Therefore:
    #
    # depth = min(
    #     distance × slope,
    #     maximum depth
    # )
    #
    # At steep angles this creates a flat bottom.
    # ========================================================

    calculated_depth = (
        distance_to_edge * slope
    )


    # Hard maximum depth
    capped_depth = np.minimum(
        calculated_depth,
        ditch_depth,
    )


    # Convert depth to negative relative elevation
    relative_ditch = (
        -capped_depth
    ).astype(np.float32)


    # Everything outside the ditch becomes NoData
    relative_ditch[
        ditch_mask == 0
    ] = np.nan


    # --------------------------------------------------------
    # ABSOLUTE RECONSTRUCTED ELEVATION
    # --------------------------------------------------------
    #
    # relative = 0 at ditch edges
    #
    # relative = -ditch_depth at deepest point
    #
    # absolute elevation =
    #
    # existing DEM + relative depth
    # --------------------------------------------------------

    absolute_ditch = (
        existing_dem_array
        + relative_ditch
    )


    absolute_ditch[
        ditch_mask == 0
    ] = np.nan


    # --------------------------------------------------------
    # INSERT DITCH INTO EXISTING DEM
    # --------------------------------------------------------

    combined_dem = (
        existing_dem_array.copy()
    )


    valid_ditch = (
        (ditch_mask == 1)
        & np.isfinite(existing_dem_array)
        & np.isfinite(absolute_ditch)
    )


    combined_dem[
        valid_ditch
    ] = absolute_ditch[
        valid_ditch
    ]


    # --------------------------------------------------------
    # OUTPUT PROFILE
    # --------------------------------------------------------

    output_profile = (
        src.profile.copy()
    )


    output_profile.update(
        driver="GTiff",
        height=raster_height,
        width=raster_width,
        transform=local_transform,
        crs=src.crs,
        count=1,
        dtype="float32",
        nodata=nodata_value,
        compress="deflate",
        predictor=3,
        tiled=True,
    )


# ============================================================
# EXPORT RELATIVE DITCH DEM
# ============================================================

relative_export = np.where(
    np.isfinite(relative_ditch),
    relative_ditch,
    nodata_value,
).astype(np.float32)


with rasterio.open(
    relative_output_tif,
    "w",
    **output_profile,
) as dst:

    dst.write(
        relative_export,
        1,
    )

    dst.set_band_description(
        1,
        "Reconstructed ditch depth relative to existing DEM",
    )

    dst.update_tags(
        surface_type="relative elevation",
        zero_reference="existing DEM surface",
        ditch_width_m=ditch_width,
        maximum_depth_m=ditch_depth,
        actual_maximum_depth_m=actual_max_depth,
        ditch_angle_degrees=ditch_angle_degrees,
        flat_bottom_width_m=flat_bottom_width,
    )


# ============================================================
# EXPORT ABSOLUTE DITCH DEM
# ============================================================

absolute_export = np.where(
    np.isfinite(absolute_ditch),
    absolute_ditch,
    nodata_value,
).astype(np.float32)


with rasterio.open(
    absolute_output_tif,
    "w",
    **output_profile,
) as dst:

    dst.write(
        absolute_export,
        1,
    )

    dst.set_band_description(
        1,
        "Absolute reconstructed ditch elevation",
    )

    dst.update_tags(
        surface_type="absolute elevation",
        reference_dem=str(existing_dem_path),
        ditch_width_m=ditch_width,
        maximum_depth_m=ditch_depth,
        actual_maximum_depth_m=actual_max_depth,
        ditch_angle_degrees=ditch_angle_degrees,
        flat_bottom_width_m=flat_bottom_width,
    )


# ============================================================
# EXPORT EXISTING DEM WITH DITCH
# ============================================================

combined_export = np.where(
    np.isfinite(combined_dem),
    combined_dem,
    nodata_value,
).astype(np.float32)


with rasterio.open(
    combined_output_tif,
    "w",
    **output_profile,
) as dst:

    dst.write(
        combined_export,
        1,
    )

    dst.set_band_description(
        1,
        "Existing DEM with reconstructed ditch",
    )

    dst.update_tags(
        surface_type="combined terrain model",
        reference_dem=str(existing_dem_path),
        ditch_width_m=ditch_width,
        maximum_depth_m=ditch_depth,
        actual_maximum_depth_m=actual_max_depth,
        ditch_angle_degrees=ditch_angle_degrees,
        flat_bottom_width_m=flat_bottom_width,
    )


# ============================================================
# RASTER-DERIVED VOLUME
# ============================================================

cell_area = (
    pixel_width
    * pixel_height
)


raster_volume = (
    np.nansum(
        -relative_ditch
    )
    * cell_area
)


# ============================================================
# ANALYTICAL CROSS-SECTION AREA
# ============================================================
#
# Two possible shapes:
#
#
# 1. V-shaped:
#
# \          /
#  \        /
#   \      /
#    \    /
#     \  /
#      \/
#
#
# 2. Depth-capped:
#
# \          /
#  \        /
#   \______/
#
#
# ============================================================

if flat_bottom_width > 0:

    # --------------------------------------------------------
    # FLAT-BOTTOMED DITCH
    # --------------------------------------------------------
    #
    # Two side triangles together:
    #
    # distance_to_max_depth × depth
    #
    # Plus:
    #
    # flat_bottom_width × depth
    # --------------------------------------------------------

    side_slopes_area = (
        distance_to_max_depth
        * ditch_depth
    )

    flat_bottom_area = (
        flat_bottom_width
        * ditch_depth
    )

    cross_section_area = (
        side_slopes_area
        + flat_bottom_area
    )

else:

    # --------------------------------------------------------
    # V-SHAPED DITCH
    # --------------------------------------------------------

    cross_section_area = (
        0.5
        * ditch_width
        * actual_max_depth
    )


analytical_volume = (
    centreline_length
    * cross_section_area
)

Ditch geometry
--------------
Width                    : 13.00 m
Side angle               : 20.00°
Maximum allowed depth    : 1.70 m
Actual maximum depth     : 1.70 m
Distance to max depth    : 4.671 m
Flat bottom width        : 3.659 m

DEM CRS     : LOCAL_CS["Amersfoort / RD New + NAP height",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Contour CRS : EPSG:28992


#### Analyse 2 Volume gracht - resultaten

In [ ]:
# ============================================================
# RESULTS
# ============================================================

print()
print("Ditch reconstruction")
print("--------------------")

print(
    f"Method                    : "
    f"{geometry_method}"
)

print(
    f"Output CRS                 : "
    f"{output_profile['crs']}"
)

print(
    f"Ditch width                : "
    f"{ditch_width:.2f} m"
)

print(
    f"Half width                 : "
    f"{half_width:.2f} m"
)

print(
    f"Side angle                 : "
    f"{ditch_angle_degrees:.2f}°"
)

print(
    f"Maximum allowed depth      : "
    f"{ditch_depth:.2f} m"
)

print(
    f"Actual maximum depth       : "
    f"{actual_max_depth:.2f} m"
)

print(
    f"Distance to maximum depth  : "
    f"{distance_to_max_depth:.3f} m"
)

print(
    f"Flat bottom width          : "
    f"{flat_bottom_width:.3f} m"
)

print(
    f"Cross-section area         : "
    f"{cross_section_area:.2f} m²"
)

print(
    f"Centreline length          : "
    f"{centreline_length:.2f} m"
)

#print(
#    f"Raster resolution          : "
#    f"{pixel_width:.2f} × "
#    f"{pixel_height:.2f} m"
#)

print(
    f"Analytical volume          : "
    f"{analytical_volume:,.0f} m³"
)

print(
    f"Raster-derived volume      : "
    f"{raster_volume:,.0f} m³"
)

print()

#print(
#    f"Relative ditch DEM         : "
#    f"{relative_output_tif}"
#)

#print(
#    f"Absolute ditch DEM         : "
#    f"{absolute_output_tif}"
#)

#print(
#    f"Existing DEM with ditch    : "
#    f"{combined_output_tif}"
#)


Ditch reconstruction
--------------------
Method                    : Side slopes capped at maximum depth; flat bottom
Output CRS                 : LOCAL_CS["Amersfoort / RD New + NAP height",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Ditch width                : 13.00 m
Half width                 : 6.50 m
Side angle                 : 20.00°
Maximum allowed depth      : 1.70 m
Actual maximum depth       : 1.70 m
Distance to maximum depth  : 4.671 m
Flat bottom width          : 3.659 m
Cross-section area         : 14.16 m²
Centreline length          : 144.06 m
Analytical volume          : 2,040 m³
Raster-derived volume      : 2,145 m³



## 2.2 Gereconstueerd volume van de heuvel
Uit gaande van een 8 meter helling vanuit de rand van de contour en een heuvel van 30° zou dit neer komen op een volume van 2121.32 m3 en een hoogte van 4,6 meter. Dan zou de hoek waarmee de gracht is gegraven rond de 20° zijn geweest en 2215 m3 opleveren, genoeg om de heuvel mee te reconstrueren.


#### Analyse 2.2 Gereconstueerd volume van de heuvel - parameters

In [10]:
# Horizontal distance over which the mound rises inward
slope_width = 8  # metres CAN BE MODIFIED
# Angle of the mound slope relative to the horizontal
slope_angle_degrees = 30 # =BETA

# Distance interval used for numerical integration.
# Smaller values are more accurate but require more calculations.
integration_step = 0.05  # metres

#### Analyse 2.2 Gereconstueerd volume van de heuvel - analyse

In [11]:
from pathlib import Path
import math

import geopandas as gpd
import numpy as np
import pandas as pd




gpkg_path = gpkg_contour

# Use the first layer when set to None
layer = layer_contour



output_folder = Path("mound_volume_results")


# ============================================================
# CHECK SETTINGS
# ============================================================

if slope_width <= 0:
    raise ValueError("slope_width must be greater than zero.")

if not 0 < slope_angle_degrees < 90:
    raise ValueError(
        "slope_angle_degrees must be between 0 and 90 degrees."
    )

if integration_step <= 0:
    raise ValueError("integration_step must be greater than zero.")


# ============================================================
# READ THE MOUND CONTOUR
# ============================================================

if layer is None:
    layers = gpd.list_layers(gpkg_path)

    if layers.empty:
        raise ValueError("No layers were found in the GeoPackage.")

    layer = layers.iloc[0]["name"]

print(f"Reading layer: {layer}")

gdf = gpd.read_file(
    gpkg_path,
    layer=layer,
)

if gdf.empty:
    raise ValueError("The selected layer contains no features.")

if gdf.crs is None:
    raise ValueError("The mound contour has no CRS.")

if not gdf.crs.is_projected:
    raise ValueError(
        "The mound contour must use a projected CRS with metres "
        "as its horizontal unit, for example EPSG:28992."
    )

# Remove missing and empty geometries
gdf = gdf[
    gdf.geometry.notna()
    & ~gdf.geometry.is_empty
].copy()

if gdf.empty:
    raise ValueError("No valid geometries remain.")

# Repair invalid geometries when supported
try:
    gdf["geometry"] = gdf.geometry.make_valid()
except AttributeError:
    gdf["geometry"] = gdf.buffer(0)

# Combine all polygon features
try:
    mound_polygon = gdf.geometry.union_all()
except AttributeError:
    mound_polygon = gdf.geometry.union_all()

if mound_polygon.is_empty:
    raise ValueError("The combined mound polygon is empty.")

if mound_polygon.geom_type not in ["Polygon", "MultiPolygon"]:
    raise TypeError(
        "The contour layer must contain polygon geometries."
    )


# ============================================================
# CALCULATE MOUND GEOMETRY
# ============================================================

angle_radians = math.radians(slope_angle_degrees)

# Vertical rise per horizontal metre
slope = math.tan(angle_radians)

# Height reached after the full slope width
maximum_height = slope_width * slope

mound_area = mound_polygon.area
mound_perimeter = mound_polygon.length


# ============================================================
# CREATE INWARD-BUFFER DISTANCES
# ============================================================

distances = np.arange(
    0,
    slope_width + integration_step,
    integration_step,
)

# Ensure the final distance is exactly slope_width
distances = distances[distances <= slope_width]

if distances[-1] < slope_width:
    distances = np.append(distances, slope_width)


# ============================================================
# CALCULATE AREA AT EACH INWARD DISTANCE
# ============================================================

remaining_areas = []

for distance in distances:

    if distance == 0:
        inward_polygon = mound_polygon
    else:
        inward_polygon = mound_polygon.buffer(-distance)

    if inward_polygon.is_empty:
        area = 0.0
    else:
        area = inward_polygon.area

    remaining_areas.append(area)

remaining_areas = np.asarray(
    remaining_areas,
    dtype=float,
)


# ============================================================
# CALCULATE VOLUME
# ============================================================
#
# At any point in the mound:
#
# height = tan(angle) × distance from the contour
#
# capped at:
#
# height = tan(angle) × slope_width
#
# The volume can therefore be calculated as:
#
# volume = tan(angle) × integral of inward-buffer area
#
# from distance 0 to slope_width.
# ============================================================

try:
    integrated_area = np.trapezoid(
        remaining_areas,
        distances,
    )
except AttributeError:
    # Compatibility with older NumPy versions
    integrated_area = np.trapz(
        remaining_areas,
        distances,
    )

mound_volume = slope * integrated_area


# ============================================================
# UPPER PLATEAU
# ============================================================

plateau_geometry = mound_polygon.buffer(-slope_width)

if plateau_geometry.is_empty:
    plateau_area = 0.0
    has_plateau = False
else:
    plateau_area = plateau_geometry.area
    has_plateau = True


# ============================================================
# ADDITIONAL STATISTICS
# ============================================================

mean_height = mound_volume / mound_area

slope_percentage = slope * 100

angle_from_vertical = 90 - slope_angle_degrees


# ============================================================
# EXPORT RESULTS
# ============================================================

output_folder.mkdir(
    parents=True,
    exist_ok=True,
)

summary = pd.DataFrame(
    [
        {
            "source_file": str(gpkg_path),
            "layer": layer,
            "crs": str(gdf.crs),
            "mound_area_m2": mound_area,
            "mound_perimeter_m": mound_perimeter,
            "slope_width_m": slope_width,
            "slope_angle_horizontal_degrees":
                slope_angle_degrees,
            "slope_angle_vertical_degrees":
                angle_from_vertical,
            "slope_percentage": slope_percentage,
            "maximum_height_m": maximum_height,
            "mean_height_m": mean_height,
            "plateau_present": has_plateau,
            "plateau_area_m2": plateau_area,
            "integration_step_m": integration_step,
            "estimated_volume_m3": mound_volume,
        }
    ]
)

summary_path = output_folder / "mound_volume_summary.csv"

summary.to_csv(
    summary_path,
    index=False,
)


# Export the mound and optional plateau
output_gpkg = output_folder / "mound_model.gpkg"

mound_output = gpd.GeoDataFrame(
    {
        "slope_width_m": [slope_width],
        "slope_angle_deg": [slope_angle_degrees],
        "maximum_height_m": [maximum_height],
        "estimated_volume_m3": [mound_volume],
    },
    geometry=[mound_polygon],
    crs=gdf.crs,
)

mound_output.to_file(
    output_gpkg,
    layer="mound_contour",
    driver="GPKG",
)

if has_plateau:
    plateau_output = gpd.GeoDataFrame(
        {
            "height_m": [maximum_height],
            "area_m2": [plateau_area],
        },
        geometry=[plateau_geometry],
        crs=gdf.crs,
    )

    plateau_output.to_file(
        output_gpkg,
        layer="upper_plateau",
        driver="GPKG",
    )

Reading layer: contour_mound


#### Analyse 2.2 Gereconstueerd volume van de heuvel - resultaten

In [12]:

print()
print("Mound-volume estimate")
print("---------------------")
print(f"Mound area             : {mound_area:,.2f} m²")
print(f"Mound perimeter        : {mound_perimeter:,.2f} m")
print(f"Slope width            : {slope_width:,.2f} m")
print(
    f"Angle from horizontal  : "
    f"{slope_angle_degrees:,.2f}°"
)
print(
    f"Angle from vertical    : "
    f"{angle_from_vertical:,.2f}°"
)
print(f"Slope                   : {slope_percentage:,.1f}%")
print(f"Maximum height         : {maximum_height:,.2f} m")
print(f"Mean modelled height   : {mean_height:,.2f} m")
print(f"Upper plateau present  : {has_plateau}")
print(f"Upper plateau area     : {plateau_area:,.2f} m²")
print(f"Estimated mound volume : {mound_volume:,.2f} m³")
print()
print(f"Summary written to     : {summary_path}")
print(f"Geometry written to    : {output_gpkg}")


Mound-volume estimate
---------------------
Mound area             : 803.70 m²
Mound perimeter        : 103.24 m
Slope width            : 8.00 m
Angle from horizontal  : 30.00°
Angle from vertical    : 60.00°
Slope                   : 57.7%
Maximum height         : 4.62 m
Mean modelled height   : 2.64 m
Upper plateau present  : True
Upper plateau area     : 184.37 m²
Estimated mound volume : 2,121.32 m³

Summary written to     : mound_volume_results\mound_volume_summary.csv
Geometry written to    : mound_volume_results\mound_model.gpkg


### Exporteren van gereconstrueerde heuvel

In [13]:
import rasterio
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt


# ============================================================
# CREATE RECONSTRUCTED DEM
# ============================================================

pixel_size = 0.5      # metres (AHN resolution)

minx, miny, maxx, maxy = mound_polygon.bounds

width = int(np.ceil((maxx - minx) / pixel_size))
height = int(np.ceil((maxy - miny) / pixel_size))

transform = rasterio.transform.from_origin(
    minx,
    maxy,
    pixel_size,
    pixel_size,
)

# Rasterize mound polygon
mask = rasterize(
    [(mound_polygon, 1)],
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype=np.uint8,
)

# Distance to the polygon edge
distance = distance_transform_edt(mask)

distance *= pixel_size

distance = np.minimum(distance, slope_width)

dem = np.tan(angle_radians) * distance

dem[mask == 0] = np.nan

output_dem = output_folder / "reconstructed_mound.tif"

profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": gdf.crs,          # same CRS as the contour
    "transform": transform,
    "nodata": -9999,
    "compress": "lzw",
}

with rasterio.open(output_dem, "w", **profile) as dst:

    out = dem.astype(np.float32)
    out[np.isnan(out)] = -9999

    dst.write(out, 1)

print(f"DEM written to {output_dem}")

DEM written to mound_volume_results\reconstructed_mound.tif


## 3.	3D visualisatie

In [14]:
from pathlib import Path
import json

import numpy as np
import rasterio
import trimesh


# ============================================================
# SETTINGS
# ============================================================

input_tifs = [
    Path(output_dem),
    Path(ahn_file),
    Path(absolute_output_tif),
    Path(combined_output_tif),
    Path(relative_output_tif),
]

output_folder = Path("ply_output")

# Use every nth raster cell:
# 1 = full resolution
# 2 = approximately 1/4 of the vertices
# 4 = approximately 1/16 of the vertices
step = 2

# Always read band 1
band_number = 1


# ============================================================
# VALIDATE INPUTS
# ============================================================

if not input_tifs:
    raise ValueError("No input TIFF files were provided.")

if step < 1:
    raise ValueError("step must be at least 1.")

for input_tif in input_tifs:
    if not input_tif.exists():
        raise FileNotFoundError(
            f"Input TIFF not found: {input_tif}"
        )

output_folder.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# DETERMINE COMMON ORIGIN
# ============================================================
#
# All rasters are assumed to already use RD New coordinates
# horizontally.
#
# AHN may report its CRS as:
#
# LOCAL_CS["Amersfoort / RD New + NAP height", ...]
#
# We therefore deliberately DO NOT reproject the rasters.
#
# A common origin is used so that all meshes line up exactly
# when imported into Blender.
# ============================================================

bounds_list = []
minimum_elevations = []

for input_tif in input_tifs:

    with rasterio.open(input_tif) as src:

        if src.count < band_number:
            raise ValueError(
                f"{input_tif} does not contain band "
                f"{band_number}."
            )

        print()
        print(f"Checking: {input_tif}")
        print(f"CRS     : {src.crs}")
        print(f"Bounds  : {src.bounds}")
        print(f"Pixel   : {src.res}")

        bounds_list.append(src.bounds)

        dem_masked = src.read(
            band_number,
            masked=True,
        )

        if dem_masked.count() == 0:
            raise ValueError(
                f"{input_tif} contains no valid "
                f"elevation values."
            )

        minimum_elevations.append(
            float(dem_masked.min())
        )


# ============================================================
# COMMON X/Y/Z ORIGIN
# ============================================================

minimum_x = min(
    bounds.left for bounds in bounds_list
)

maximum_x = max(
    bounds.right for bounds in bounds_list
)

minimum_y = min(
    bounds.bottom for bounds in bounds_list
)

maximum_y = max(
    bounds.top for bounds in bounds_list
)

# Centre of combined raster extent
x_origin = (minimum_x + maximum_x) / 2
y_origin = (minimum_y + maximum_y) / 2

# Lowest elevation in all rasters
z_origin = min(minimum_elevations)


print()
print("Common origin")
print("-------------")
print(f"X origin: {x_origin:.3f}")
print(f"Y origin: {y_origin:.3f}")
print(f"Z origin: {z_origin:.3f}")


# ============================================================
# FUNCTION: TIFF TO PLY
# ============================================================

def tif_to_ply(
    input_tif,
    output_ply,
    x_origin,
    y_origin,
    z_origin,
    step=1,
):
    """
    Convert band 1 of a GeoTIFF to a PLY terrain mesh.

    The raster is assumed to already use RD New coordinates
    horizontally and metres vertically.

    No CRS transformation is performed.

    A common X/Y/Z origin is subtracted so multiple PLY files
    line up correctly when imported into Blender.
    """

    # --------------------------------------------------------
    # READ RASTER
    # --------------------------------------------------------

    with rasterio.open(input_tif) as src:

        original_crs = src.crs
        transform = src.transform

        dem_masked = src.read(
            band_number,
            masked=True,
        )

    # --------------------------------------------------------
    # SUBSAMPLE
    # --------------------------------------------------------

    dem_masked = dem_masked[
        ::step,
        ::step,
    ]

    dem = dem_masked.filled(
        np.nan
    ).astype(np.float32)

    rows, cols = dem.shape

    if rows < 2 or cols < 2:
        raise ValueError(
            f"{input_tif} is too small after "
            f"applying step={step}."
        )

    valid = np.isfinite(dem)

    if not valid.any():
        raise ValueError(
            f"{input_tif} contains no valid "
            f"elevation values."
        )


    # --------------------------------------------------------
    # ORIGINAL RASTER CELL INDICES
    # --------------------------------------------------------
    #
    # If step = 2:
    #
    # new raster column 0 -> original column 0
    # new raster column 1 -> original column 2
    # new raster column 2 -> original column 4
    #
    # This preserves the correct geographic coordinates.
    # --------------------------------------------------------

    raster_rows = np.arange(rows) * step
    raster_cols = np.arange(cols) * step

    col_grid, row_grid = np.meshgrid(
        raster_cols,
        raster_rows,
    )


    # --------------------------------------------------------
    # CALCULATE CELL-CENTRE COORDINATES
    # --------------------------------------------------------
    #
    # Uses the raster's affine transform directly.
    #
    # No reprojection is performed.
    # --------------------------------------------------------

    x = (
        transform.c
        + (col_grid + 0.5) * transform.a
        + (row_grid + 0.5) * transform.b
    )

    y = (
        transform.f
        + (col_grid + 0.5) * transform.d
        + (row_grid + 0.5) * transform.e
    )


    # --------------------------------------------------------
    # SHIFT TO COMMON BLENDER ORIGIN
    # --------------------------------------------------------

    x_local = x - x_origin
    y_local = y - y_origin
    z_local = dem - z_origin


    # --------------------------------------------------------
    # CREATE VERTICES
    # --------------------------------------------------------

    vertices = np.column_stack(
        (
            x_local.ravel(),
            y_local.ravel(),
            z_local.ravel(),
        )
    ).astype(np.float32)


    # --------------------------------------------------------
    # CREATE TRIANGULAR FACES
    # --------------------------------------------------------

    vertex_ids = np.arange(
        rows * cols,
        dtype=np.int64,
    ).reshape(rows, cols)

    top_left = vertex_ids[:-1, :-1]
    top_right = vertex_ids[:-1, 1:]
    bottom_left = vertex_ids[1:, :-1]
    bottom_right = vertex_ids[1:, 1:]


    # Triangle 1
    faces_1 = np.column_stack(
        (
            top_left.ravel(),
            bottom_left.ravel(),
            top_right.ravel(),
        )
    )


    # Triangle 2
    faces_2 = np.column_stack(
        (
            top_right.ravel(),
            bottom_left.ravel(),
            bottom_right.ravel(),
        )
    )


    faces = np.vstack(
        (
            faces_1,
            faces_2,
        )
    )


    # --------------------------------------------------------
    # REMOVE FACES TOUCHING NODATA
    # --------------------------------------------------------

    valid_flat = valid.ravel()

    valid_faces = valid_flat[faces].all(
        axis=1
    )

    faces = faces[valid_faces]

    if len(faces) == 0:
        raise ValueError(
            f"No valid faces could be created for "
            f"{input_tif}."
        )


    # --------------------------------------------------------
    # CREATE MESH
    # --------------------------------------------------------

    mesh = trimesh.Trimesh(
        vertices=vertices,
        faces=faces,
        process=False,
    )


    # Remove vertices that are no longer referenced after
    # NoData faces were removed.
    mesh.remove_unreferenced_vertices()


    # Make normals consistent
    trimesh.repair.fix_normals(mesh)


    # --------------------------------------------------------
    # EXPORT PLY
    # --------------------------------------------------------

    mesh.export(output_ply)


    # --------------------------------------------------------
    # RETURN INFORMATION
    # --------------------------------------------------------

    return {
        "input_tif": str(input_tif),
        "output_ply": str(output_ply),
        "original_crs": str(original_crs),
        "horizontal_reference": "RD New / EPSG:28992",
        "vertices": int(len(mesh.vertices)),
        "faces": int(len(mesh.faces)),
        "step": step,
    }


# ============================================================
# PROCESS ALL TIFF FILES
# ============================================================

results = []

for input_tif in input_tifs:

    output_ply = (
        output_folder
        / f"{input_tif.stem}.ply"
    )

    print()
    print(f"Converting: {input_tif}")

    result = tif_to_ply(
        input_tif=input_tif,
        output_ply=output_ply,
        x_origin=x_origin,
        y_origin=y_origin,
        z_origin=z_origin,
        step=step,
    )

    results.append(result)

    print(f"Output PLY : {output_ply}")
    print(
        f"Vertices   : "
        f"{result['vertices']:,}"
    )
    print(
        f"Faces      : "
        f"{result['faces']:,}"
    )


# ============================================================
# SAVE COORDINATE OFFSETS
# ============================================================
#
# These values allow Blender coordinates to be converted back
# to the original RD/NAP coordinates.
#
# RD X = Blender X + x_origin
# RD Y = Blender Y + y_origin
# NAP Z = Blender Z + z_origin
# ============================================================

metadata = {

    "horizontal_reference": (
        "RD New / EPSG:28992"
    ),

    "common_origin": {
        "x_origin": x_origin,
        "y_origin": y_origin,
        "z_origin": z_origin,
    },

    "coordinate_conversion": {
        "rd_x": "blender_x + x_origin",
        "rd_y": "blender_y + y_origin",
        "nap_z": "blender_z + z_origin",
    },

    "meshes": results,
}


metadata_path = (
    output_folder
    / "ply_coordinate_offsets.json"
)


with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        metadata,
        file,
        indent=4,
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print("Conversion completed")
print("--------------------")
print(
    f"TIFF files processed : "
    f"{len(results)}"
)
print(
    "Horizontal reference : "
    "RD New / EPSG:28992"
)
print(
    f"Common X origin       : "
    f"{x_origin:.3f}"
)
print(
    f"Common Y origin       : "
    f"{y_origin:.3f}"
)
print(
    f"Common Z origin       : "
    f"{z_origin:.3f}"
)
print(
    f"Metadata              : "
    f"{metadata_path}"
)


Checking: mound_volume_results\reconstructed_mound.tif
CRS     : LOCAL_CS["Amersfoort / RD New",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Bounds  : BoundingBox(left=151231.69763444096, bottom=600732.4255786766, right=151264.69763444096, top=600766.4255786766)
Pixel   : (0.5, 0.5)

Checking: data\AHN5_M_151000_600000.TIF
CRS     : LOCAL_CS["Amersfoort / RD New + NAP height",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Bounds  : BoundingBox(left=151000.0, bottom=600000.0, right=152000.0, top=601000.0)
Pixel   : (0.5, 0.5)

Checking: ditch_results\reconstructed_ditch_absolute.tif
CRS     : LOCAL_CS["Amersfoort / RD New + NAP height",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Bounds  : BoundingBox(left=151218.5, bottom=600720.0, right=151277.5, top=600779.5)
Pixel   : (0.5, 0.5)

Checking: ditch_results\existing_dem_with_ditch.tif
CRS     : LOCAL_CS["Amersfoort /

<img src=".//img//3D_resultaat.png" alt="" style="display: inline-block; height: 1000; width: 1250; vertical-align: text-bottom; margin: 0 0.25rem;" />